# Fase C — Consultas SQL analíticas
**Proyecto 1 — Bodega de delitos Colombia 2025 (DIJIN)** · Grupo 7

Seis consultas SQL ejecutadas contra la bodega cargada en la Fase B. Cada una responde una
pregunta de negocio explícita, muestra el resultado en tabla y se interpreta debajo.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

RAIZ_PROYECTO = Path.cwd().parent if Path.cwd().name == "entrega" else Path.cwd()
load_dotenv(RAIZ_PROYECTO / ".env")
DATABASE_URL = os.getenv("DATABASE_URL") or f"sqlite:///{(RAIZ_PROYECTO / 'datos' / 'bodega_delitos.db').as_posix()}"
engine = create_engine(DATABASE_URL)
pd.set_option("display.max_columns", None)
print("Conectado a:", engine.url.render_as_string(hide_password=True))

Conectado a: sqlite:///c%3A/Users/Usuario/Documents/Asignaturas/Ciencia de datos/proyecto_1/Grupo-7-p1/datos/bodega_delitos.db


## Consulta 1 — ¿Cómo evolucionó el número de casos mes a mes durante 2025?

*(serie de tiempo, JOIN explícito hecho ↔ dim_tiempo)*

In [11]:
q1 = """
SELECT t.anio, t.mes, t.nombre_mes, SUM(h.cantidad) AS total_casos
FROM hecho_delito h
JOIN dim_tiempo t ON h.fecha_id = t.fecha_id
GROUP BY t.anio, t.mes, t.nombre_mes
ORDER BY t.mes
"""
r1 = pd.read_sql(q1, engine)
r1


,anio,mes,nombre_mes,total_casos
0,2025,1,January,38792
1,2025,2,February,37054
2,2025,3,March,40735
3,2025,4,April,40275
4,2025,5,May,38187
5,2025,6,June,39353
6,2025,7,July,43712
7,2025,8,August,43509
8,2025,9,September,44205
9,2025,10,October,46611


**Interpretación.** El volumen mensual de casos oscila entre ~30.900 (diciembre) y ~46.600
(octubre), sin una tendencia claramente creciente o decreciente a lo largo del año: hay un
repunte sostenido entre julio y octubre y una caída marcada en noviembre-diciembre. La caída de
diciembre podría reflejar un efecto de reporte (datos aún no consolidados al cierre del año) más
que una baja real en la actividad delictiva, algo que vale la pena señalar como limitación.

## Consulta 2 — ¿Cómo se distribuye cada tipo de delito por departamento y trimestre del año?

*(GROUP BY con 3 dimensiones: departamento × tipo de delito × trimestre)*

In [12]:
q2 = """
SELECT g.departamento, d.tipo_delito, t.trimestre, SUM(h.cantidad) AS total_casos
FROM hecho_delito h
JOIN dim_geografia g ON h.geo_id = g.geo_id
JOIN dim_tipo_delito d ON h.delito_id = d.delito_id
JOIN dim_tiempo t ON h.fecha_id = t.fecha_id
GROUP BY g.departamento, d.tipo_delito, t.trimestre
ORDER BY total_casos DESC
LIMIT 15
"""
r2 = pd.read_sql(q2, engine)
r2


,departamento,tipo_delito,trimestre,total_casos
0,CUNDINAMARCA,HURTO A PERSONAS,3,38956
1,CUNDINAMARCA,HURTO A PERSONAS,1,36253
2,CUNDINAMARCA,HURTO A PERSONAS,2,35847
3,CUNDINAMARCA,HURTO A PERSONAS,4,28798
4,CUNDINAMARCA,VIOLENCIA INTRAFAMILIAR,3,19030
5,CUNDINAMARCA,VIOLENCIA INTRAFAMILIAR,2,14419
6,CUNDINAMARCA,VIOLENCIA INTRAFAMILIAR,4,13009
7,CUNDINAMARCA,VIOLENCIA INTRAFAMILIAR,1,12752
8,ANTIOQUIA,HURTO A PERSONAS,4,10611
9,ANTIOQUIA,HURTO A PERSONAS,3,8855


**Interpretación.** Cundinamarca (que incluye Bogotá) concentra, por amplio margen, las
combinaciones departamento-delito-trimestre con más casos: domina tanto en hurto a personas
como en violencia intrafamiliar en los 4 trimestres. El tercer trimestre (jul-sep) es el pico
de hurto a personas en Cundinamarca, mientras que en Antioquia y Valle el patrón se concentra
más hacia el cuarto trimestre.

## Consulta 3 — ¿Cuáles son los 3 municipios con más casos dentro de cada tipo de delito?

*(función de ventana `RANK() OVER (PARTITION BY ...)`)*

In [13]:
q3 = """
WITH casos_municipio AS (
    SELECT d.tipo_delito, g.municipio, g.departamento, SUM(h.cantidad) AS total_casos
    FROM hecho_delito h
    JOIN dim_tipo_delito d ON h.delito_id = d.delito_id
    JOIN dim_geografia g ON h.geo_id = g.geo_id
    GROUP BY d.tipo_delito, g.municipio, g.departamento
)
SELECT tipo_delito, municipio, departamento, total_casos, ranking
FROM (
    SELECT *, RANK() OVER (PARTITION BY tipo_delito ORDER BY total_casos DESC) AS ranking
    FROM casos_municipio
)
WHERE ranking <= 3
ORDER BY tipo_delito, ranking
"""
r3 = pd.read_sql(q3, engine)
r3


,tipo_delito,municipio,departamento,total_casos,ranking
0,HOMICIDIO INTENCIONAL,BOGOTÁ D.C. (CT),CUNDINAMARCA,1173,1
1,HOMICIDIO INTENCIONAL,CALI (CT),VALLE,1060,2
2,HOMICIDIO INTENCIONAL,BARRANQUILLA (CT),ATLÁNTICO,428,3
3,HOMICIDIOS EN ACCIDENTE DE TRÁNSITO,BOGOTÁ D.C. (CT),CUNDINAMARCA,577,1
4,HOMICIDIOS EN ACCIDENTE DE TRÁNSITO,MEDELLÍN (CT),ANTIOQUIA,505,2
5,HOMICIDIOS EN ACCIDENTE DE TRÁNSITO,CALI (CT),VALLE,225,3
6,HURTO A PERSONAS,BOGOTÁ D.C. (CT),CUNDINAMARCA,123364,1
7,HURTO A PERSONAS,MEDELLÍN (CT),ANTIOQUIA,25117,2
8,HURTO A PERSONAS,CALI (CT),VALLE,18079,3
9,HURTO A RESIDENCIAS,BOGOTÁ D.C. (CT),CUNDINAMARCA,5976,1


**Interpretación.** Bogotá D.C. aparece en el puesto 1 en prácticamente todos los tipos de
delito, lo cual es esperable dado su tamaño poblacional. El caso de Terrorismo tiene 4 filas en
vez de 3 porque hay un empate en el tercer lugar (`RANK()` -a diferencia de `ROW_NUMBER()`- deja
explícito ese empate en vez de romperlo arbitrariamente). Fuera de Bogotá, ningún otro municipio
se repite en el top 3 de más de dos delitos distintos, lo que sugiere que la concentración
geográfica del delito varía bastante según el tipo.

## Consulta 4 — ¿Qué armas o medios se usan con más frecuencia en cada tipo de delito?

*(JOIN explícito hecho ↔ dim_arma_medio ↔ dim_tipo_delito)*

In [14]:
q4 = """
SELECT a.arma_medio, d.tipo_delito, SUM(h.cantidad) AS total_casos
FROM hecho_delito h
JOIN dim_arma_medio a ON h.arma_id = a.arma_id
JOIN dim_tipo_delito d ON h.delito_id = d.delito_id
GROUP BY a.arma_medio, d.tipo_delito
ORDER BY total_casos DESC
LIMIT 15
"""
r4 = pd.read_sql(q4, engine)
r4


,arma_medio,tipo_delito,total_casos
0,SIN EMPLEO DE ARMAS,HURTO A PERSONAS,186542
1,SIN EMPLEO DE ARMAS,VIOLENCIA INTRAFAMILIAR,121754
2,ARMA DE FUEGO,HURTO A PERSONAS,43054
3,ARMA BLANCA / CORTOPUNZANTE,HURTO A PERSONAS,42565
4,SIN EMPLEO DE ARMAS,HURTO A RESIDENCIAS,18488
5,CONTUNDENTES,VIOLENCIA INTRAFAMILIAR,18250
6,ARMA DE FUEGO,HOMICIDIO INTENCIONAL,10922
7,NO REPORTADO,HURTO A PERSONAS,9009
8,CONTUNDENTES,HURTO A PERSONAS,8089
9,NO REPORTADO,HOMICIDIOS EN ACCIDENTE DE TRÁNSITO,4528


**Interpretación.** "Sin empleo de armas" es, con diferencia, la categoría más frecuente
(domina en hurto a personas y violencia intrafamiliar), lo que tiene sentido porque son delitos
que no siempre requieren un arma física. El arma de fuego y el arma blanca/cortopunzante
concentran los casos de homicidio intencional, confirmando su rol central en los delitos más
violentos.

## Consulta 5 — ¿Qué perfil de víctima (género y grupo de edad) predomina en cada tipo de delito?

*(relación bivariada género × grupo de edad, filtrando víctimas con género reportado)*

In [15]:
q5 = """
SELECT d.tipo_delito, v.genero, v.grupo_edad, SUM(h.cantidad) AS total_casos
FROM hecho_delito h
JOIN dim_victima v ON h.victima_id = v.victima_id
JOIN dim_tipo_delito d ON h.delito_id = d.delito_id
WHERE v.genero != 'NO REPORTADO'
GROUP BY d.tipo_delito, v.genero, v.grupo_edad
ORDER BY total_casos DESC
LIMIT 15
"""
r5 = pd.read_sql(q5, engine)
r5


,tipo_delito,genero,grupo_edad,total_casos
0,HURTO A PERSONAS,MASCULINO,ADULTOS,161483
1,HURTO A PERSONAS,FEMENINO,ADULTOS,124883
2,VIOLENCIA INTRAFAMILIAR,FEMENINO,ADULTOS,97476
3,VIOLENCIA INTRAFAMILIAR,MASCULINO,ADULTOS,30973
4,HURTO A RESIDENCIAS,MASCULINO,ADULTOS,14203
5,HOMICIDIO INTENCIONAL,MASCULINO,ADULTOS,12251
6,HURTO A RESIDENCIAS,FEMENINO,ADULTOS,11248
7,HOMICIDIOS EN ACCIDENTE DE TRÁNSITO,MASCULINO,ADULTOS,5702
8,VIOLENCIA INTRAFAMILIAR,FEMENINO,MENORES,4130
9,VIOLENCIA INTRAFAMILIAR,MASCULINO,MENORES,3289


**Interpretación.** El perfil de víctima cambia claramente según el delito: en hurto a
personas predominan los adultos masculinos (161.483 casos) seguidos de cerca por adultas
(124.883), mientras que en violencia intrafamiliar la proporción se invierte fuertemente hacia
mujeres adultas (97.476 vs 30.973 de hombres). Esto es consistente con la naturaleza del delito y
sugiere que las estrategias de prevención deberían diferenciarse por tipo de delito y perfil de
víctima.

## Consulta 6 — ¿Los delitos ocurren con más frecuencia en fin de semana o entre semana?

*(distribución univariada por día de la semana)*

In [16]:
q6 = """
SELECT t.dia_semana, t.es_fin_de_semana, SUM(h.cantidad) AS total_casos
FROM hecho_delito h
JOIN dim_tiempo t ON h.fecha_id = t.fecha_id
GROUP BY t.dia_semana, t.es_fin_de_semana
ORDER BY total_casos DESC
"""
r6 = pd.read_sql(q6, engine)
r6


,dia_semana,es_fin_de_semana,total_casos
0,Saturday,1,73109
1,Sunday,1,70046
2,Friday,0,69772
3,Wednesday,0,68822
4,Monday,0,67047
5,Thursday,0,65912
6,Tuesday,0,65383


**Interpretación.** Sábado (73.109) y domingo (70.046) son, individualmente, los días con
más casos, pero como hay 5 días entre semana contra 2 de fin de semana, en total entre semana
sigue acumulando más casos en términos absolutos. En promedio por día, el fin de semana
(~71.578/día) supera claramente a los días hábiles (~67.387/día), lo que sugiere una asociación
entre tiempo libre/ocio nocturno y mayor incidencia delictiva.